# Format Converter

Конвертер генераторного JSONL с inline-тегами вида `<NAME>...</NAME>` в span-format:

```json
{
  "text": "чистый текст без тегов",
  "entities": [
    {"start": 0, "end": 10, "label": "NAME", "text": "..."}
  ]
}
```

Важный принцип: истина для span-разметки — это содержимое тегов прямо в `text`, а не `used_entities`. Это нужно, потому что модель иногда склоняет/нормализует значения при вставке.

In [1]:
from pathlib import Path
import json
import re
from collections import Counter

INPUT_PATH = Path("../outputs/synthesized_pii.jsonl")
OUTPUT_PATH = Path("../outputs/converted_synthesized_pii_spans.jsonl")

TAG_RE = re.compile(r"<([A-Z0-9_]+)>(.*?)</\1>", flags=re.DOTALL)

print(f"Input:  {INPUT_PATH.resolve()}")
print(f"Output: {OUTPUT_PATH.resolve()}")

Input:  /Users/artemzmailov/Desktop/kitoboy-PII/synthetic_data_generation/synthesizer_agent/outputs/synthesized_pii.jsonl
Output: /Users/artemzmailov/Desktop/kitoboy-PII/synthetic_data_generation/synthesizer_agent/outputs/converted_synthesized_pii_spans.jsonl


## Load JSONL

In [2]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            item["_jsonl_line"] = line_no
            rows.append(item)
    return rows

rows = load_jsonl(INPUT_PATH)
print(f"Loaded rows: {len(rows)}")

Loaded rows: 1825


## Converter

`convert_inline_tags_to_spans` идет по тегам слева направо, копирует обычный текст без изменений, а содержимое тега записывает в чистый текст и одновременно создает span.

In [3]:
def convert_inline_tags_to_spans(marked_text: str) -> tuple[str, list[dict]]:
    clean_parts = []
    entities = []
    cursor = 0
    clean_len = 0

    for match in TAG_RE.finditer(marked_text):
        label = match.group(1)
        value = match.group(2)

        before = marked_text[cursor:match.start()]
        clean_parts.append(before)
        clean_len += len(before)

        start = clean_len
        clean_parts.append(value)
        clean_len += len(value)
        end = clean_len

        entities.append({
            "start": start,
            "end": end,
            "label": label,
            "text": value,
        })

        cursor = match.end()

    tail = marked_text[cursor:]
    clean_parts.append(tail)

    clean_text = "".join(clean_parts)

    for ent in entities:
        extracted = clean_text[ent["start"]:ent["end"]]
        if extracted != ent["text"]:
            raise ValueError(
                "Span check failed: "
                f"expected={ent['text']!r}, extracted={extracted!r}, "
                f"start={ent['start']}, end={ent['end']}"
            )

    return clean_text, entities

# Smoke check
clean_text, entities = convert_inline_tags_to_spans("Меня зовут <NAME>Иван</NAME>, почта <EMAIL>a@b.ru</EMAIL>.")
print(clean_text)
print(entities)

Меня зовут Иван, почта a@b.ru.
[{'start': 11, 'end': 15, 'label': 'NAME', 'text': 'Иван'}, {'start': 23, 'end': 29, 'label': 'EMAIL', 'text': 'a@b.ru'}]


## Convert Dataset

In [4]:
def convert_item(item: dict) -> dict:
    clean_text, entities = convert_inline_tags_to_spans(item["text"])
    return {
        "text": clean_text,
        "entities": entities,
    }

converted = [convert_item(item) for item in rows]

entity_counts = Counter(
    ent["label"]
    for item in converted
    for ent in item["entities"]
)

print(f"Converted rows: {len(converted)}")
print(f"Total spans: {sum(entity_counts.values())}")
print("Entity counts:")
for label, count in sorted(entity_counts.items()):
    print(f"  - {label}: {count}")

Converted rows: 1825
Total spans: 2779
Entity counts:
  - ADDRESS: 305
  - BANK_CARD: 310
  - EMAIL: 310
  - NAME: 309
  - ORGANIZATION: 309
  - PASSPORT_RF: 310
  - PHONE_NUMBER: 309
  - TELEGRAM: 308
  - VK: 309


## Diagnostics

Смотрим строки, где в тексте нет тегов, и сравниваем число тегов с `used_entities` только как debug-сигнал. Это не блокирует конвертацию.

In [5]:
no_span_rows = [item for item in converted if not item["entities"]]
print(f"Rows without spans: {len(no_span_rows)}")

count_mismatches = []
for src, dst in zip(rows, converted):
    used_count = len(src.get("used_entities", []))
    span_count = len(dst["entities"])
    if used_count != span_count:
        count_mismatches.append({
            "jsonl_line": src.get("_jsonl_line"),
            "used_entities_count": used_count,
            "span_count": span_count,
            "used_entities": src.get("used_entities", []),
            "entities": dst["entities"],
            "text": src.get("text", ""),
        })

print(f"Rows where used_entities count != tag span count: {len(count_mismatches)}")

if count_mismatches:
    count_mismatches[:5]

Rows without spans: 0
Rows where used_entities count != tag span count: 0


## Preview

In [7]:
def show_converted(i: int):
    item = converted[i]
    source = rows[i]
    print("=" * 100)
    print(f"Converted item #{i}, source jsonl_line={source.get('_jsonl_line')}")
    print("\nTEXT:\n")
    print(item["text"])
    print("\nENTITIES:\n")
    print(json.dumps(item["entities"], ensure_ascii=False, indent=2))
    print("\nSOURCE USED_ENTITIES:\n")
    print(json.dumps(source.get("used_entities", []), ensure_ascii=False, indent=2))

show_converted(0)


Converted item #0, source jsonl_line=1

TEXT:

Я учусь в школе. Есть несколько друзей. Там. И лучшая подруга. Ну как лучшая. Для меня лучшая. Насчет ее отношения ко мне не знаю. Но я очень ценю наше общение. Каждая прогулка с ней для меня как праздник. Но, увы, это бывает не часто. Чаще она гуляет с нашими одноклассниками. А влиться в их компанию нн получается. Ну не принимают они меня.

Я даже пытался написать ей в телеграм https://t.me/maria_ivanova_7017, но она так и не ответила. Может, я что-то не так сделал? Или она просто не хочет общаться? Не знаю. Если кто-то из вас её увидит, скажите, что я пытался связаться. Или пусть она сама напишет мне на номер +7 (921) 999-44-66. Мне бы хотя бы понять, в чём дело.

ENTITIES:

[
  {
    "start": 382,
    "end": 413,
    "label": "TELEGRAM",
    "text": "https://t.me/maria_ivanova_7017"
  },
  {
    "start": 619,
    "end": 637,
    "label": "PHONE_NUMBER",
    "text": "+7 (921) 999-44-66"
  }
]

SOURCE USED_ENTITIES:

[
  {
    "key": "TEL

## Save JSONL

Сохраняем рядом с накопительным датасетом в файл с приставкой `converted_`. Если файл уже существует, он просто перезаписывается.

In [9]:
def save_jsonl(rows: list[dict], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for item in rows:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"Saved rows: {len(rows)}")
    print(f"Saved to: {path.resolve()}")

SAVE_OUTPUT = True

if SAVE_OUTPUT:
    save_jsonl(converted, OUTPUT_PATH)
else:
    print("SAVE_OUTPUT=False: файл пока не записан. Поставь True, когда захочешь сохранить.")


Saved rows: 1825
Saved to: /Users/artemzmailov/Desktop/kitoboy-PII/synthetic_data_generation/synthesizer_agent/outputs/converted_synthesized_pii_spans.jsonl
